***OOP Project***

Contains:

1. Seq Class (added len and eq overloaders)
2. DNA Class (my method: open reading frame, can apply to RNA too if edit start/stop codons)
3. RNA Class
4. Protein Class

In [23]:
import re
import doctest

In [24]:
standard_code = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L", "UCU": "S",
    "UCC": "S", "UCA": "S", "UCG": "S", "UAU": "Y", "UAC": "Y",
    "UAA": "*", "UAG": "*", "UGA": "*", "UGU": "C", "UGC": "C",
    "UGG": "W", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P", "CAU": "H",
    "CAC": "H", "CAA": "Q", "CAG": "Q", "CGU": "R", "CGC": "R",
    "CGA": "R", "CGG": "R", "AUU": "I", "AUC": "I", "AUA": "I",
    "AUG": "M", "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K", "AGU": "S",
    "AGC": "S", "AGA": "R", "AGG": "R", "GUU": "V", "GUC": "V",
    "GUA": "V", "GUG": "V", "GCU": "A", "GCC": "A", "GCA": "A",
    "GCG": "A", "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G"}

kyte_doolittle={'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,
                'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,
                'T':-0.7,'V':4.2,'W':-0.9,'X':0,'Y':-1.3}

aa_mol_weights={'A':89.09,'C':121.15,'D':133.1,'E':147.13,'F':165.19,
                'G':75.07,'H':155.16,'I':131.17,'K':146.19,'L':131.17,
                'M':149.21,'N':132.12,'P':115.13,'Q':146.15,'R':174.2,
                'S':105.09,'T':119.12,'V':117.15,'W':204.23,'X':0,'Y':181.19}

In [54]:
class Seq:
    """
    Constructor: 
        Input: (sequence, gene name, species)
        Uses the string functions upper and strip to clean self.sequence
        Adds an empty list 'self.kmers'
    Methods: 
       a. 'print_record' returns the sequence
       b. 'print' overload: str function to return species, gene name : sequence
       c. 'make_kmers' makes overlapping kmers of a given length from self.sequence,
           appends to self.kmers. Default kmer parameter=3.
        d. 'fasta' returns a fasta formatted string
    
    >>> myseq=Seq("ATATAG","my_gene","H.sapiens")
    >>> print(myseq)
    H.sapiens, my_gene : ATATAG
    >>> print(myseq.print_record())
    ATATAG
    >>> print(myseq.gene)
    my_gene
    >>> print(myseq.species)
    H.sapiens
    >>> print(myseq.fasta())
    >H.sapiens my_gene
    ATATAG
    >>> print(myseq.make_kmers())
    ['ATA', 'TAT', 'ATA', 'TAG']


    >>> dna1 = DNA("ATGC", "x", "x", "x")
    >>> dna2 = DNA("ATGC", "x", "x", "x")
    >>> dna3 = DNA("ATGA", "x", "x", "x")
    >>> rna1 = RNA("ATGC", "x", "x", "x")
    >>> print(dna1 == dna2)
    True
    >>> print(dna1 == dna3)
    False
    >>> print(dna1 == rna1)
    False
    >>> dna4 = DNA("atgc", "x", "x", "x")
    >>> print(dna1 == dna4)
    True

    """
    
    def __init__(self,sequence,gene,species,**kwargs):
        self.sequence=sequence.strip().upper()
        self.gene=gene
        self.species=species
        self.kmers = []
  
    def __str__(self):
        return f"{self.species}, {self.gene} : {self.sequence}"
        
    def print_record(self):
        return self.sequence

    def make_kmers(self, k=3):
        self.kmers = []
        for i in range(len(self.sequence) - k + 1):
            self.kmers.append(self.sequence[i:i+k])
        return self.kmers
        
    def fasta(self): #exports fasta-formatted string, but not a file
        return f">{self.species} {self.gene}\n{self.sequence}" 

    def __len__(self):
        return len(self.sequence)
        
    def __eq__(self, other):
        if type(self) is not type(other):
            return False
        return self.sequence == other.sequence


class DNA(Seq):
    """
    Class DNA inherits Seq. 
    
    Constructor: adds gene_id and **kwargs (sequence, gene name, species, gene_id)
        re.sub to change any non nucleotide characters in self.sequence into an 'N'
        - re.sub('[^ATGCU]','N',sequence) will change any character that is not a
        capital A, T, G, C or U into an N.
    Methods: 
        a. 'analysis' returns the total count number of Gs and Cs
        b. 'print_info' prints geneid and sequence 
        c. 'reverse_complement' returns the reverse complement of self.sequence
        d. 'six_frames' that returns all 6 frames of self.sequence
            This include the 3 forward frames, and the 3 reverse complement frames
        e.  'find_orfs' finds the open reading frames in the sequence (forward)

    >>> d=DNA("GATCTC","my_dna","D.terebrans","AX5667.2")
    >>> print(d)
    D.terebrans, my_dna : GATCTC
    >>> d.source="Mexico"
    >>> print(d.source)
    Mexico
    >>> print(d.analysis())
    3
    >>> print(d.print_info())
    AX5667.2 : GATCTC
    >>> print(d.reverse_complement())
    GAGATC
    >>> print(d.six_frames())
    ['GATCTC', 'ATCTC', 'TCTC', 'GAGATC', 'AGATC', 'GATC']
    
    >>> d=DNA("AAATGAAATTTTAAATGTAG","my_dna","species","gene_id")
    >>> print(d.find_orfs())
    ['ATGAAATTTTAA', 'ATGTAG']

    """
    
    def __init__(self, sequence, gene, species, gene_id, **kwargs):
        super().__init__(sequence, gene, species, **kwargs)
        self.gene_id = gene_id
        self.sequence = re.sub('[^ATGCU]', 'N', self.sequence)
   
    def print_info(self): # why does it need an empty space at beginning?
        return f"{self.gene_id} : {self.sequence}"
        
    def analysis(self):
        return sum(1 for x in self.sequence if x in ['G', 'C']) #adds 1 every time theres G or C and returns the total

    def reverse_complement(self):
        complement = {'A':'T', 'T':'A', 'G':'C', 'C':'G', 'U':'A', 'N':'N'}
        rev_comp = ""
        for base in reversed(self.sequence): #reverse first
            rev_comp += complement[base]
        return rev_comp

    def six_frames(self):
        frames = []
    #Forward frames
        for i in range(3):
            frames.append(self.sequence[i:])
     #Reverse complement
        rev_comp = self.reverse_complement()
    #Reverse frames
        for i in range(3):
            frames.append(rev_comp[i:])
        return frames

    def find_orfs(self):
        start_codon = "ATG"
        stop_codons = {"TAA", "TAG", "TGA"}
        seq = self.sequence
        orfs = []

        # check all 3 reading frames
        for frame in range(3):
            i = frame
            while i < len(seq) - 2:
                codon = seq[i:i+3]
                if codon == start_codon:  # found start, now search for stop
                    for j in range(i, len(seq) - 2, 3):
                        codon = seq[j:j+3]
                        if codon in stop_codons:
                            orfs.append(seq[i:j+3])
                            i = j + 3  # move past this ORF
                            break
                    else:
                        # no stop codon found?
                        i += 3
                else:
                    i += 3
        return orfs

***RNA Class***

*RNA Constructor:*

Inherits DNA class
Use the super() function (see DNA Class example).

1. Automatically change all Ts to Us in self.sequence.
2. Add self.codons equals to an empty list

*Methods:*

1. Add make_codons which breaks the self.sequence into
   3 letter codons and appends these codons to self.codons
   unless they are less than 3 letters long.
3. Add translate which uses the Global Variable standard_code below to
   translate the codons in self.codons and returns a protein sequence.

In [43]:
class RNA(DNA):
    """
    Constructor:
        Inherits DNA class (sequence, gene name, species, gene_id)
        Automatically changes all Ts to Us in self.sequence
        Adds self.codons equals to an empty list
    
    Methods:
        a. 'make_codon' breaks the self.sequence into 3 letter codons 
        and appends these codons to self.codons
        unless they are less than 3 letters long.
        
        b. 'translate' uses the Global Variable standard_code to
        translate codons in self.codons and returns a protein sequence
  
    >>> x = RNA("ATGCGTAA", "katG", "E.coli", "gene123")
    >>> print(x.make_codons())
    ['AUG', 'CGU']
    >>> print(x.translate())
    MR
    
    """
   
    def __init__(self, sequence, gene, species, gene_id, **kwargs):
        super().__init__(sequence, gene, species, gene_id, **kwargs)
        self.sequence = self.sequence.replace('T', 'U') # convert T → U
        self.codons = []

    def make_codons(self):
        self.codons = []
        for i in range(0, len(self.sequence), 3):
            codon = self.sequence[i:i+3]
            if len(codon) == 3:
                self.codons.append(codon)
        return self.codons

    def translate(self):
        protein = ""
        for codon in self.codons:
            protein += standard_code.get(codon, 'X') # if > else X
        return protein

***Protein Class***

*Constructor:*
1. Use the super() function (see DNA Class example).
2. Use re.sub to change any non LETTER characters in self.sequence into an 'X'.

*Methods:*
The next 2 methods use a kyte_doolittle and the aa_mol_weights dictionaries.

1. Add total_hydro, which return the sum of
   the total hydrophobicity of a self.sequence
2. Add mol_weight, which returns the total molecular weight of
   the protein sequence assigned to the protein object.

In [46]:
class Protein(Seq):
    """
    Constructor:
        Uses re.sub to change any non LETTER characters in self.sequence into an 'X'

    Methods:
        The next 2 methods use a kyte_doolittle and the aa_mol_weights dictionaries.
        a. 'total_hydro' returns the sum of the total hydrophobicity of a self.sequence
        b. 'mol_weight' returns the total molecular weight of
           the protein sequence assigned to the protein object.
    
    >>> p = Protein("MVLSPADKTN", "hemoglobin", "human")
    >>> print("Sequence:", p.sequence)
    Sequence: MVLSPADKTN
    >>> print("Total hydrophobicity:", p.total_hydro())
    Total hydrophobicity: -2.3000000000000007
    >>> print("Molecular weight:", p.mol_weight())
    Molecular weight: 1237.37
    >>> print(p.fasta()) 
    >human hemoglobin
    MVLSPADKTN
    """
    
    def __init__(self, sequence, gene, species, **kwargs):
        super().__init__(sequence, gene, species, **kwargs)
        self.sequence = re.sub('[^A-Z]', 'X', self.sequence)

    def total_hydro(self):
        total = 0
        for aa in self.sequence:
            total += kyte_doolittle[aa]
        return total

    def mol_weight(self):
        total = 0
        for aa in self.sequence:
            total += aa_mol_weights[aa]
        return total

In [56]:
# run doctests
doctest.testmod(verbose=True)

Trying:
    d=DNA("GATCTC","my_dna","D.terebrans","AX5667.2")
Expecting nothing
ok
Trying:
    print(d)
Expecting:
    D.terebrans, my_dna : GATCTC
ok
Trying:
    d.source="Mexico"
Expecting nothing
ok
Trying:
    print(d.source)
Expecting:
    Mexico
ok
Trying:
    print(d.analysis())
Expecting:
    3
ok
Trying:
    print(d.print_info())
Expecting:
    AX5667.2 : GATCTC
ok
Trying:
    print(d.reverse_complement())
Expecting:
    GAGATC
ok
Trying:
    print(d.six_frames())
Expecting:
    ['GATCTC', 'ATCTC', 'TCTC', 'GAGATC', 'AGATC', 'GATC']
ok
Trying:
    d=DNA("AAATGAAATTTTAAATGTAG","my_dna","species","gene_id")
Expecting nothing
ok
Trying:
    print(d.find_orfs())
Expecting:
    ['ATGAAATTTTAA', 'ATGTAG']
ok
Trying:
    p = Protein("MVLSPADKTN", "hemoglobin", "human")
Expecting nothing
ok
Trying:
    print("Sequence:", p.sequence)
Expecting:
    Sequence: MVLSPADKTN
ok
Trying:
    print("Total hydrophobicity:", p.total_hydro())
Expecting:
    Total hydrophobicity: -2.300000000000000

TestResults(failed=0, attempted=34)